In [3]:
# ── Cell 1: Install ────────────────────────────────────────────────────────
import subprocess, os

PKGS = "/kaggle/input/datasets/dennisfong/nvidia-nemotron-offline-packages/offline_packages"
MAMBA_PKGS = "/kaggle/input/datasets/mayukh18/nemotron-packages"

# Core packages
subprocess.run(
    f"pip install -q --no-index --find-links {PKGS} "
    "datasets trl peft transformers accelerate safetensors",
    shell=True, check=True
)

# mamba_ssm and causal_conv1d
subprocess.run(
    f"pip install -q --no-deps {MAMBA_PKGS}/causal_conv1d-1.6.1+cu12torch2.10cxx11abiTRUE-cp312-cp312-linux_x86_64.whl",
    shell=True, check=True
)
subprocess.run(
    f"pip install -q --no-deps {MAMBA_PKGS}/mamba_ssm-2.3.1+cu12torch2.10cxx11abiTRUE-cp312-cp312-linux_x86_64.whl",
    shell=True, check=True
)

print("Done")

Done


In [4]:
# ── Cell 2: Triton + env fixes (critical for Blackwell) ───────────────────
import os, sys, stat, shutil
import torch
import torch.nn.functional as F

# Bypass broken Triton rmsnorm on Blackwell
def _pure_rmsnorm_fn(x, weight, bias=None, z=None, eps=1e-5,
                     group_size=None, norm_before_gate=True, upcast=True):
    dtype = x.dtype
    if upcast:
        x = x.float()
    variance = x.pow(2).mean(-1, keepdim=True)
    x_normed = x * torch.rsqrt(variance + eps)
    out = x_normed * weight.float()
    if bias is not None:
        out = out + bias.float()
    if z is not None:
        out = out * F.silu(z.float())
    return out.to(dtype)

for name, mod in list(sys.modules.items()):
    if hasattr(mod, 'rmsnorm_fn'):
        mod.rmsnorm_fn = _pure_rmsnorm_fn

# Copy ptxas-blackwell to writable location
src = "/kaggle/usr/lib/notebooks/ryanholbrook/nvidia-utility-script/triton/backends/nvidia/bin/ptxas-blackwell"
dst = "/tmp/ptxas-blackwell"
if os.path.exists(src):
    shutil.copy2(src, dst)
    os.chmod(dst, os.stat(dst).st_mode | stat.S_IEXEC | stat.S_IXGRP | stat.S_IXOTH)

    import triton.backends.nvidia as nv_backend
    src_bin = os.path.join(os.path.dirname(nv_backend.__file__), "bin")
    dst_bin = "/tmp/triton_nvidia_bin"
    shutil.copytree(src_bin, dst_bin, dirs_exist_ok=True)
    for f in os.listdir(dst_bin):
        fp = os.path.join(dst_bin, f)
        if os.path.isfile(fp):
            os.chmod(fp, os.stat(fp).st_mode | stat.S_IEXEC | stat.S_IXGRP | stat.S_IXOTH)

    nv_backend.__file__ = os.path.join(dst_bin, "..", "__init__.py")
    os.environ["TRITON_PTXAS_PATH"] = dst
    os.environ["TRITON_PTXAS_BLACKWELL_PATH"] = dst

print(f"GPU : {torch.cuda.get_device_name(0)}")
print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB")

GPU : NVIDIA RTX PRO 6000 Blackwell Server Edition
VRAM: 102.0 GB


In [7]:
# ── Cell 3: Config ─────────────────────────────────────────────────────────
LORA_RANK    = 32
LORA_ALPHA   = 16
LORA_DROPOUT = 0.05
MAX_SEQ_LEN  = 4096
NUM_EPOCHS   = 1
GRAD_ACCUM   = 4
LR           = 2e-4
OUTPUT_DIR   = "/kaggle/working/adapter"
os.makedirs(OUTPUT_DIR, exist_ok=True)

In [ ]:
# ── Cell 4: Load model ─────────────────────────────────────────────────────
import kagglehub, torch, sys
from transformers import AutoModelForCausalLM, AutoTokenizer

MODEL_PATH = kagglehub.model_download(
    "metric/nemotron-3-nano-30b-a3b-bf16/transformers/default"
)
print(f"Model path: {MODEL_PATH}")

tokenizer = AutoTokenizer.from_pretrained(MODEL_PATH, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    MODEL_PATH,
    device_map="auto",
    trust_remote_code=True,
    dtype=torch.bfloat16,
    attn_implementation="eager",
)
print(f"Model loaded. Vocab size: {len(tokenizer)}")

# Disable broken Mamba CUDA fast path on Blackwell
for name, mod in sys.modules.items():
    if "modeling_nemotron_h" in name and hasattr(mod, "is_fast_path_available"):
        mod.is_fast_path_available = False
        print(f"Patched {name}: is_fast_path_available = False")

Model path: /kaggle/input/models/metric/nemotron-3-nano-30b-a3b-bf16/transformers/default/1


Loading weights:   0%|          | 0/6243 [00:00<?, ?it/s]

In [ ]:
# ── Cell 5: Apply LoRA ─────────────────────────────────────────────────────
from peft import LoraConfig, get_peft_model, TaskType

lora_config = LoraConfig(
    r=LORA_RANK,
    lora_alpha=LORA_ALPHA,
    target_modules="all-linear",
    lora_dropout=LORA_DROPOUT,
    bias="none",
    task_type=TaskType.CAUSAL_LM,
)
model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

In [ ]:
# ── Cell 6: Load dataset ───────────────────────────────────────────────────
from datasets import load_dataset

raw_ds = load_dataset(
    "json",
    data_files={
        "train": "/kaggle/input/datasets/tadityasasidhar/sft-hf-dataset/train.jsonl",
    }
)["train"]

print(f"Loaded {len(raw_ds)} examples")
print("Sample:", raw_ds[0]["text"][:200])

In [ ]:
# ── Cell 7: Train ──────────────────────────────────────────────────────────
import triton.backends.nvidia.compiler as nv_compiler
from trl import SFTTrainer, SFTConfig

os.environ["TRITON_PTXAS_BLACKWELL_PATH"] = "/tmp/ptxas-blackwell"
nv_compiler.get_ptxas_version = lambda arch: "12.0"

training_args = SFTConfig(
    output_dir=OUTPUT_DIR,
    per_device_train_batch_size=1,
    gradient_accumulation_steps=GRAD_ACCUM,
    num_train_epochs=NUM_EPOCHS,
    learning_rate=LR,
    logging_steps=10,
    bf16=True,
    max_grad_norm=1.0,
    optim="adamw_torch",
    lr_scheduler_type="cosine",
    warmup_steps=50,
    save_strategy="no",
    report_to="none",
    dataset_text_field="text",
    max_length=MAX_SEQ_LEN,
    packing=True,
    gradient_checkpointing=True,
    gradient_checkpointing_kwargs={"use_reentrant": True},
)

trainer = SFTTrainer(
    model=model,
    train_dataset=raw_ds,
    processing_class=tokenizer,
    args=training_args,
)

print("Starting training...")
trainer.train()
print("Training complete.")

In [ ]:
! pip list


In [ ]:
import os, zipfile, json
from safetensors.torch import load_file, save_file

model.save_pretrained(SFT_OUT)
tokenizer.save_pretrained(SFT_OUT)

# Fix adapter_config.json — remove auto_mapping
config_path = os.path.join(SFT_OUT, "adapter_config.json")
with open(config_path) as f:
    config = json.load(f)
config["auto_mapping"] = None
with open(config_path, "w") as f:
    json.dump(config, f, indent=2)

# Verify tensor keys (no renaming needed)
st_path = os.path.join(SFT_OUT, "adapter_model.safetensors")
tensors = load_file(st_path)
print(f"Saved {len(tensors)} tensors")
for k in list(tensors.keys())[:5]:
    print(" ", k)

# Create README
with open(os.path.join(SFT_OUT, "README.md"), "w") as f:
    f.write("---\nbase_model: /kaggle/input/models/metric/nemotron-3-nano-30b-a3b-bf16/transformers/default/1\nlibrary_name: peft\n---\n# Nemotron LoRA Adapter\n")

# Package — include notebook and README
zip_path = os.path.join(SFT_OUT, "submission.zip")
files_to_zip = ["adapter_config.json", "adapter_model.safetensors", "README.md"]

# Find notebook
import glob
nbs = glob.glob("/kaggle/working/*.ipynb") + glob.glob("/kaggle/input/**/*.ipynb", recursive=True)
if nbs:
    import shutil
    nb_dst = os.path.join(SFT_OUT, "__notebook__.ipynb")
    shutil.copy(nbs[0], nb_dst)
    files_to_zip.append("__notebook__.ipynb")

with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as zf:
    for fname in files_to_zip:
        fpath = os.path.join(SFT_OUT, fname)
        if os.path.exists(fpath):
            zf.write(fpath, fname)
            print(f"Added: {fname}")

print(f"\nsubmission.zip: {os.path.getsize(zip_path)/1e6:.1f} MB")
print("Contents:", zipfile.ZipFile(zip_path).namelist())